In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
# first-time use execution
!pip install biopython

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/antigen/')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()
## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --upgrade

In [ ]:
import pymesh


In [ ]:
!pip install trimesh

In [ ]:
!pip install pyglet

In [ ]:
!pip install pyvista

In [ ]:
!python --version

In [ ]:
!pip install ipyvolume

In [ ]:
!pip install pywavefront

In [ ]:
import pywavefront
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO
from scipy.spatial import Delaunay
import trimesh
import itertools
from scipy.spatial import ConvexHull
import pyvista as pv
from pymol import cmd, finish_launching
import matplotlib.pyplot as plt
from collections import defaultdict
import os

In [ ]:
local_path = 'drive/MyDrive/antigen/'
obj_filename = '1a2y_B_A_C.obj'
pdb_filename = '1a2y_B_A_C.pdb'
obj_folder_path = 'drive/MyDrive/antigen/pdb_obj/'
if not os.path.exists(obj_folder_path):
  os.makedirs(obj_folder_path)
  print(f"Folder '{obj_folder_path}' was created.")
else:
  print(f"Folder '{obj_folder_path}' already exists.")

In [ ]:
cmd.load(local_path + pdb_filename, 'pdb_filename')
cmd.hide('everything', 'pdb_filename')
cmd.show('surface', 'pdb_filename')
cmd.set_view((
     1,    0,    0,
     0,    1,    0,
     0,    0,    1,
     0,    0,    0,
     0,    0,    0,
     0,    300,  1
))
cmd.save(obj_folder_path + obj_filename, 'pdb_filename', format='obj')
# cmd.quit() crashed everytime

In [ ]:
# Read the .obj file
mesh = pywavefront.Wavefront(obj_folder_path + obj_filename, collect_faces=True)

In [ ]:
obj_vertices = mesh.vertices
len(obj_vertices)

In [ ]:
len(set(obj_vertices))

In [ ]:
obj_coords = np.array([v for v in set(obj_vertices)])
obj_mean_curvature = np.array([np.nan] * len(obj_coords))
obj_guassian_curvature = np.array([np.nan] * len(obj_coords))
obj_curvature = np.array([np.nan] * len(obj_coords))

In [ ]:
len(obj_coords)

In [ ]:
#@title Archive: surface.obj vertices with **flag (3 conditions)**
# for idx in range(len(obj_coords)):
#   current_coords = obj_coords[idx]
#   dist_coords = obj_coords - current_coords

#   dists = np.linalg.norm(dist_coords, axis=1)
#   sorted_indices = np.argsort(dists)

#   nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
#   vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

#   cloud = pv.PolyData(vertices)
#   triangulated = cloud.delaunay_2d()
#   shell = triangulated.extract_geometry()

#   extracted_vertices = np.array(shell.points)
#   extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
#   extracted_faces = np.array([face for face in extracted_faces if 0 in face])

#   mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
#   mesh, info = pymesh.remove_isolated_vertices(mesh)

#   mesh.add_attribute("vertex_mean_curvature")
#   H = mesh.get_attribute("vertex_mean_curvature")
#   mean_H = H[0]

#   mesh.add_attribute("vertex_gaussian_curvature")
#   K = mesh.get_attribute("vertex_gaussian_curvature")
#   gaussian_K = K[0]

#   elem = np.square(mean_H) - gaussian_K
#   flag = True

#   if gaussian_K > 0 and mean_H < 0:
#     flag = False
#   elif gaussian_K < 0 and mean_H > 0:
#     flag = False
#   elif gaussian_K > 0 and mean_H > 0:
#     flag = False

#   if elem < 0:
#     elem = 1e-8
#   elem = np.array(elem)

#   k1 = mean_H + np.sqrt(elem)
#   k2 = mean_H - np.sqrt(elem)


#   si = (k1 + k2) / (k1 - k2)
#   si = np.arctan(si) * (-2 / np.pi)

#   if flag:
#     si = -si

#   obj_mean_curvature[idx] = mean_H
#   obj_guassian_curvature[idx] = gaussian_K
#   obj_curvature[idx] = si

# import ipyvolume as ipv
# from matplotlib import cm

# colors = cm.RdBu_r((obj_curvature + 1) / 2)

# x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
# y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
# z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

# center = obj_coords.mean(axis=0)
# centered_coords = obj_coords - center

# ipv.figure(width=600, height=500)
# scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
# ipv.xlim(x_min, x_max)
# ipv.ylim(y_min, y_max)
# ipv.zlim(z_min, z_max)

# ipv.show()

In [ ]:
#@title Archive: surface.obj vertices with **flag (1 condition)**
# for idx in range(len(obj_coords)):
#   current_coords = obj_coords[idx]
#   dist_coords = obj_coords - current_coords

#   dists = np.linalg.norm(dist_coords, axis=1)
#   sorted_indices = np.argsort(dists)

#   nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
#   vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

#   cloud = pv.PolyData(vertices)
#   triangulated = cloud.delaunay_2d()
#   shell = triangulated.extract_geometry()

#   extracted_vertices = np.array(shell.points)
#   extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
#   extracted_faces = np.array([face for face in extracted_faces if 0 in face])

#   mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
#   mesh, info = pymesh.remove_isolated_vertices(mesh)

#   mesh.add_attribute("vertex_mean_curvature")
#   H = mesh.get_attribute("vertex_mean_curvature")
#   mean_H = H[0]

#   mesh.add_attribute("vertex_gaussian_curvature")
#   K = mesh.get_attribute("vertex_gaussian_curvature")
#   gaussian_K = K[0]

#   elem = np.square(mean_H) - gaussian_K
#   flag = True

#   if gaussian_K > 0 and mean_H < 0:
#     flag = False

#   if elem < 0:
#     elem = 1e-8
#   elem = np.array(elem)

#   k1 = mean_H + np.sqrt(elem)
#   k2 = mean_H - np.sqrt(elem)

#   si = (k1 + k2) / (k1 - k2)
#   si = np.arctan(si) * (-2 / np.pi)

#   if flag:
#     si = -si

#   obj_mean_curvature[idx] = mean_H
#   obj_guassian_curvature[idx] = gaussian_K
#   obj_curvature[idx] = si


# import ipyvolume as ipv
# from matplotlib import cm

# colors = cm.RdBu_r((obj_curvature + 1) / 2)

# x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
# y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
# z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

# center = obj_coords.mean(axis=0)
# centered_coords = obj_coords - center

# ipv.figure(width=600, height=500)
# scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
# ipv.xlim(x_min, x_max)
# ipv.ylim(y_min, y_max)
# ipv.zlim(z_min, z_max)

# ipv.show()

In [ ]:
#@title Calculate surface.obj vertex curvature

# x_median = np.median(obj_coords[:, 0])
# y_median = np.median(obj_coords[:, 1])
# z_median = np.median(obj_coords[:, 2])
# centroid = np.array([x_median, y_median, z_median])


# 15 seems to be the best neighbor number
n_neighbors = 15

# Calculate centroid by mean
centroid = np.mean(obj_coords, axis=0)

for idx in range(len(obj_coords)):
  current_coords = obj_coords[idx]
  dist_coords = obj_coords - current_coords

  dists = np.linalg.norm(dist_coords, axis=1)
  sorted_indices = np.argsort(dists)

  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

  cloud = pv.PolyData(vertices)
  triangulated = cloud.delaunay_2d()
  shell = triangulated.extract_geometry()

  extracted_vertices = np.array(shell.points)
  extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
  extracted_faces = np.array([face for face in extracted_faces if 0 in face])

  mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
  mesh, info = pymesh.remove_isolated_vertices(mesh)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  gaussian_K = K[0]

  elem = np.square(mean_H) - gaussian_K

  if elem < 0:
    elem = 1e-8
  elem = np.array(elem)

  k1 = mean_H + np.sqrt(elem)
  k2 = mean_H - np.sqrt(elem)

  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (-2 / np.pi)

  dist_to_centroid = np.linalg.norm(current_coords - centroid)
  neighbor_dists_to_centroid = [np.linalg.norm(obj_coords[i] - centroid) for i in nearest_neighbors_idx]
  avg_neighbor_dist_to_centroid = np.mean(neighbor_dists_to_centroid)

  curvature_sign_by_distance = -1 if dist_to_centroid < avg_neighbor_dist_to_centroid else 1

  curvature_sign = curvature_sign_by_distance

  if curvature_sign > 0:
    si = abs(si)
  elif curvature_sign < 0:
    si = -abs(si)

  obj_mean_curvature[idx] = mean_H
  obj_guassian_curvature[idx] = gaussian_K
  obj_curvature[idx] = si

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', local_path + '1a2y_B_A_C.pdb')
model = structure[0]
chain = model['C']

# Get all atoms in the pdb file
atoms = [atom for atom in chain.get_atoms()]

# Remove HETATM atom
atoms = [atom for atom in chain.get_atoms() if atom.get_parent().id[0] == " "]
atom_curvature = [None] * len(atoms)

# atoms = [atom for atom in chain.get_atoms()]
atom_coords = np.array([atom.coord for atom in atoms])

print('ATOM (NON-HETATM) number: ', len(atoms))

In [ ]:
# Read tsv file
# df_tsv = pd.read_csv(os.path.join(local_path, '1a2y_B_A_C_info.tsv'),delimiter='\t')
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# print('length of tsv: ', len(df_tsv))
# surface_residues = df_tsv[df_tsv['SaSaRate'] > 0.2]
# print('Atom number (SaSaRate>0.2) : ', len(surface_residues))
# surface_residues_coords = surface_residues[['SCx', 'SCy', 'SCz']].values.astype(float)

In [ ]:
#@title Plot the surface colored by shape index
import ipyvolume as ipv
from matplotlib import cm

colors = cm.RdBu_r((obj_curvature + 1) / 2)

x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

center = obj_coords.mean(axis=0)
centered_coords = obj_coords - center

centroid_centered = centroid - center
# coords_center = coords - coords.mean(axis=0)
coords_center = atom_coords - center
# coords_center = coords - coords.mean(axis=0)
# centered_surface_residues_coords = surface_residues_coords - center

ipv.figure(width=600, height=500)
scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
ipv.scatter(centroid_centered[0], centroid_centered[1], centroid_centered[2], size=150, marker="sphere")

ipv.scatter(coords_center[:, 0], coords_center[:, 1], coords_center[:, 2], size=100, color='yellow', marker='sphere')
# ipv.scatter(centered_surface_residues_coords[:, 0], centered_surface_residues_coords[:, 1], centered_surface_residues_coords[:, 2], size=100, color='yellow', marker='sphere')

ipv.xlim(x_min, x_max)
ipv.ylim(y_min, y_max)
ipv.zlim(z_min, z_max)

ipv.show()

In [ ]:
#@title Select the mean curvature of the closest n vertices
n_neighbors = 10
for idx in range(len(atom_coords)):
  current_atom_coords = atom_coords[idx]
  dist_coords = obj_coords - current_atom_coords
  dists = np.linalg.norm(dist_coords, axis=1)

  nearest_neighbors_idx = np.argsort(dists)[:n_neighbors]
  atom_curvature[idx] = np.mean(obj_curvature[nearest_neighbors_idx])

In [ ]:
#@title Select the mean curvature of vertices within distance_threshold

# distance_threshold = 1.75
# for idx in range(len(atom_coords)):
#   current_atom_coords = atom_coords[idx]
#   dist_coords = obj_coords - current_atom_coords
#   dists = np.linalg.norm(dist_coords, axis=1)

#   sorted_indices = np.argsort(dists)
#   nearest_neighbors_idx = sorted_indices[dists[sorted_indices] < distance_threshold]


#   if len(nearest_neighbors_idx) == 0:
#     atom_curvature[idx] = 0
#   else:
#     atom_curvature[idx] = np.mean(obj_curvature[nearest_neighbors_idx])

In [ ]:
def select_and_update(structure, curvatures):
  idx = 0
  for model in structure:
    for chain in model:
      if chain.id == 'C':
        for i, residue in enumerate(chain):
          for j, atom in enumerate(residue):
            if atom.get_parent().id[0] == " ":
              curvature_value = atom_curvature[idx]
              atom.set_bfactor(curvature_value)

In [ ]:
# Create a new PDBIO object for saving the modified structure
io = PDBIO()

# Save the updated chain C structure
output_path = f'mean_{n_neighbors}dist_nearest_one_point_curvature.pdb'
io.set_structure(structure)
select_and_update(structure, atom_curvature)
io.save(output_path)